# 00 · Ambiente, dati e protocollo

**Tesi:** *Valutazione di una strategia di market making mediante dati limit order book L2, L3 e L4.*

**Domanda di ricerca.** Una strategia di market making semplice, attiva solo quando il mercato è calmo, resta profittevole se il backtest smette di "indovinare" le esecuzioni con regole sui dati aggregati (L2) e le simula invece con la coda reale degli ordini (L3 e L4)?

Questo notebook prepara il terreno: controlla l'ambiente e i dati, descrive la strategia e il protocollo con cui è stata valutata, ed esegue i test automatici.

| Notebook | Cosa risponde |
|---|---|
| 00 · setup | I dati sono quelli dichiarati? Come è fatto l'esperimento? |
| 01 · FI-2010 | Come si comporta un book L2 e perché le regole L2 sbagliano? |
| 02 · backtest L2 | Quanto cambia il risultato cambiando solo la regola di esecuzione L2? (E1, E2) |
| 03 · ITCH | Il book ricostruito dai messaggi Nasdaq è corretto? |
| 04 · backtest L3 | Con la coda reale degli ordini la strategia guadagna? (E3) |
| 05 · Hyperliquid | Vale anche su un mercato crypto con dati L4? (E4) |
| 06 · risultati | Risposta finale, costi e latenza (E5), prestazioni, glossario |

I termini tecnici sono spiegati nel **glossario** in fondo al notebook 06.

Per rigenerare tutti i risultati: `scripts/download_data.py`, poi `mm-backtest ingest`, `mm-backtest pipeline` e `scripts/validate_notebooks.py`.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/market_making_tesi").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from market_making_tesi import report as R
from market_making_tesi.report import it, pct
R.setup_plots()
FIGURES = ROOT / "results" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
np.random.seed(0)
import platform
from importlib.metadata import version
environment = {"python": platform.python_version(), "sistema": platform.platform(),
               **{n: version(n) for n in ["numpy", "pandas", "pyarrow", "matplotlib"]}}
(ROOT / "results" / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
display(pd.Series(environment, name="versione").to_frame())

,versione
python,3.14.6
sistema,Windows-11-10.0.26300-SP0
numpy,2.5.4
pandas,3.0.6
pyarrow,26.0.0
matplotlib,3.11.2


## 1. I dati sono quelli dichiarati?

Ogni file scaricato è registrato in `data/manifest.json` con la fonte, la data di accesso e un'impronta digitale (SHA-256): se anche un solo byte cambiasse, l'impronta non corrisponderebbe più. La cella ricalcola le impronte di tutti i file (qualche minuto, circa 40 GB).

In [2]:
import hashlib
manifest = json.loads((ROOT / "data" / "manifest.json").read_text(encoding="utf-8"))

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            digest.update(block)
    return digest.hexdigest()

rows = []
for key, entry in sorted(manifest.items()):
    path = ROOT / key
    ok = path.exists() and path.stat().st_size == entry["bytes"] and sha256(path) == entry["sha256"]
    rows.append({"dataset": key.split("/")[2], "GB": entry["bytes"] / 1e9, "verificato": ok,
                 "fonte": entry["source"].split("/")[2], "accesso": entry["accessed"]})
files = pd.DataFrame(rows)
summary = files.groupby("dataset").agg(file=("GB", "size"), GB=("GB", "sum"), tutti_verificati=("verificato", "all"),
                                       fonte=("fonte", "first"), accesso=("accesso", "max"))
display(summary.style.format({"GB": "{:.2f}"}))
assert files.verificato.all(), "file mancanti o alterati: rieseguire scripts/download_data.py"

,file,GB,tutti_verificati,fonte,accesso
dataset,,,,,
fi2010,10,0.94,True,etsin.fairdata.fi,2026-10-14
hyperliquid,12,0.79,True,zenodo.org,2026-10-14
itch,7,31.01,True,emi.nasdaq.com,2026-10-14


## 2. La strategia in una pagina

La strategia è volutamente semplice e **identica in tutti gli esperimenti**; cambia solo il modo di decidere se i suoi ordini vengono eseguiti.

1. **Misura la volatilità**: a ogni aggiornamento del book calcola quanto si è mosso il prezzo medio (midpoint) nell'ultimo tratto di mercato (una *finestra mobile*).
2. **Opera solo se il mercato è calmo**: se la volatilità supera una soglia cancella i suoi ordini, chiude l'inventario e aspetta.
3. **Quota una coppia**: con il mercato calmo e senza posizioni aperte inserisce un ordine di acquisto al miglior prezzo in acquisto (best bid) e uno di vendita al miglior prezzo in vendita (best ask). Se entrambi vengono eseguiti, guadagna la differenza: lo *spread*.
4. **Chiude il rischio**: se dopo un certo tempo è stato eseguito un solo lato, cancella l'altro e vende o compra a mercato per tornare a inventario zero.

La tabella riporta i parametri fissi per ciascun dataset; finestra, soglia e durata degli ordini sono invece scelti in validazione (punto 3).

In [3]:
from market_making_tesi.config import StrategyConfig
from market_making_tesi.pipeline import DATASETS, session_list
cfg = {n: StrategyConfig.from_json(ROOT / "config" / f"{n}.json") for n in ["fi2010", "itch", "hyperliquid"]}
display(pd.DataFrame({
    "quantità per lato": {n: f"{it(c.quote_size, 2 if c.quote_size < 1 else 0)} {u}" for (n, c), u in
                          zip(cfg.items(), ["unità", "azioni", "BTC"])},
    "inventario massimo": {n: it(c.max_abs_inventory, 2 if c.max_abs_inventory < 1 else 0) for n, c in cfg.items()},
    "volatilità misurata": {"fi2010": "per osservazione", "itch": "midpoint ogni secondo",
                            "hyperliquid": "per evento (i dati non hanno orari)"},
    "rendimenti": {n: "logaritmici" if c.return_type == "log" else "differenze (prezzi normalizzati)" for n, c in cfg.items()},
}).rename(index=R.DATASET_LABELS))

,quantità per lato,inventario massimo,volatilità misurata,rendimenti
FI-2010,1 unità,1,per osservazione,differenze (prezzi normalizzati)
Nasdaq ITCH,100 azioni,100,midpoint ogni secondo,logaritmici
Hyperliquid BTC,"0,01 BTC","0,01",per evento (i dati non hanno orari),logaritmici


## 3. Il protocollo: calibrazione, validazione, test

Per non "barare" scegliendo i parametri sugli stessi dati su cui si misura il risultato, le sessioni sono divise **in ordine cronologico**:

| Fase | A cosa serve | Esempio (ITCH) |
|---|---|---|
| Calibrazione | stimare le soglie di volatilità (quanto è "calmo" il mercato) | 30/01/2019 |
| Validazione | scegliere finestra, soglia e durata degli ordini tra poche combinazioni | 27/03/2019 |
| Test | misurare il risultato, senza più toccare nulla | 5 giornate da luglio 2019 a gennaio 2020 |

Una *sessione* è un titolo in una giornata (FI-2010, ITCH) o un'ora di negoziazione (Hyperliquid).

In [4]:
rows = []
for name, spec in DATASETS.items():
    for session, symbol, _ in session_list(ROOT, spec):
        phase = "calibrazione" if session in spec.calibration else "validazione" if session in spec.validation else "test"
        rows.append({"dataset": R.DATASET_LABELS[name], "fase": phase, "sessione": session, "simbolo": symbol})
sessions = pd.DataFrame(rows)
order = {"calibrazione": 0, "validazione": 1, "test": 2}
table = (sessions.groupby(["dataset", "fase"]).agg(sessioni=("sessione", "size"), dal=("sessione", "min"),
                                                    al=("sessione", "max"), strumenti=("simbolo", "nunique"))
         .reset_index().sort_values(["dataset", "fase"], key=lambda s: s.map(order) if s.name == "fase" else s))
display(table.set_index(["dataset", "fase"]))

sessioni          dal           al  strumenti
dataset         fase                                                       
FI-2010         calibrazione         5        day01        day01          5
                validazione         10        day02        day03          5
                test                35        day04        day10          5
Hyperliquid BTC calibrazione         1  20251201T08  20251201T08          1
                validazione          1  20251201T14  20251201T14          1
                test                 4  20251202T08  20251203T14          1
Nasdaq ITCH     calibrazione         3   2019-01-30   2019-01-30          3
                validazione          3   2019-03-27   2019-03-27          3
                test                15   2019-07-30   2020-01-30          3

## 4. Il codice funziona?

I test automatici verificano ogni componente su casi costruiti a mano di cui si conosce il risultato esatto: lettura dei file, ricostruzione del book, regole di esecuzione, coda degli ordini, latenza, contabilità e metriche. Poi il motore viene eseguito sui piccoli campioni di `data/samples`, che devono riprodurre valori noti.

In [5]:
import subprocess
check = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"],
                       cwd=ROOT, capture_output=True, text=True, timeout=600)
log = check.stdout + check.stderr
(ROOT / "results" / "test_report.txt").write_text(log, encoding="utf-8")
assert check.returncode == 0, log
ran = next(line for line in log.splitlines() if line.startswith("Ran "))
from market_making_tesi.experiments import run_samples
samples, _ = run_samples(ROOT, ROOT / "results" / "samples")
assert samples.fill_maker.tolist() == [22, 5, 2, 2, 0]
R.show_box("Esito dei controlli", [
    f"{ran.split()[1]} test automatici superati.",
    "I cinque campioni di controllo riproducono i valori attesi (fill e P&L).",
    "Tutti i file grezzi corrispondono al manifesto.",
])

> **Esito dei controlli**
>
> - 47 test automatici superati.
> - I cinque campioni di controllo riproducono i valori attesi (fill e P&L).
> - Tutti i file grezzi corrispondono al manifesto.